In [2]:
import pandas as pd

# 1. 加载并预处理行业数据
industry = pd.read_csv('./data/股票行业分类/industry.csv')
industry['InfoPublDate'] = pd.to_datetime(industry['InfoPublDate'])
# 统一列名为 'symbol' 以便匹配，并确保 6 位代码格式
industry = industry.rename(columns={'SecuCode': 'symbol'})
industry['symbol'] = industry['symbol'].astype(str).str.zfill(6)
# merge_asof 必须要求右表是按时间排好序的
industry = industry.sort_values('InfoPublDate')

def check_industry_stability(file_path, index_name):
    # 2. 读取成分股权重数据
    df = pd.read_parquet(file_path)
    df['date'] = pd.to_datetime(df['date'])
    df['symbol'] = df['symbol'].astype(str).str.zfill(6)
    
    # 筛选目标时间范围
    df = df[(df['date'] >= '2016-01-01') & (df['date'] <= '2018-12-31')]
    
    # merge_asof 要求左表也必须按时间排序
    df = df.sort_values('date')
    
    # 3. 使用 merge_asof 匹配行业
    # 统一使用 'symbol' 列进行关联
    merged = pd.merge_asof(
        df, 
        industry, 
        left_on='date', 
        right_on='InfoPublDate', 
        by='symbol', 
        direction='backward'
    )
    
    # 4. 按月统计行业总数
    # 将日期转换为月份（Period类型，如 '2016-01'）
    merged['month'] = merged['date'].dt.to_period('M')
    
    # 统计每个月出现的唯一行业(tag)数量
    monthly_counts = merged.groupby('month')['tag'].nunique()
    
    # 打印结果报告
    print(f"--- {index_name} 行业稳定性检查 ---")
    print(f"各月行业总数最小/最大值: {monthly_counts.min()} / {monthly_counts.max()}")
    
    if monthly_counts.nunique() == 1:
        print(f"✅ 结果：{index_name} 的行业总数在 2016-2018 期间始终为 {monthly_counts.iloc[0]} 种。")
    else:
        print(f"⚠️ 警告：{index_name} 的行业总数发生过波动！")
        # 打印出具体的变动情况供查看
        print(monthly_counts.value_counts().to_string())
    
    print("-" * 30)
    return monthly_counts

# 执行检查
counts_300 = check_industry_stability('./data/指数权重/index_wt_000300.parquet', '沪深300')
counts_500 = check_industry_stability('./data/指数权重/index_wt_000905.parquet', '中证500')

--- 沪深300 行业稳定性检查 ---
各月行业总数最小/最大值: 28 / 29
⚠️ 警告：沪深300 的行业总数发生过波动！
tag
29    19
28    17
------------------------------
--- 中证500 行业稳定性检查 ---
各月行业总数最小/最大值: 28 / 29
⚠️ 警告：中证500 的行业总数发生过波动！
tag
29    19
28    17
------------------------------
